In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
from pathlib import Path

d:\Projects\GenAI Labs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()

True

In [3]:
# Step 1: Load PDF
data_path = Path("../data")
file_path = data_path / "Reinforcement Learning from Human Feedback.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

In [4]:
# Step 2: Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
chunks = splitter.split_documents(documents)

In [5]:
print(chunks[0].page_content)

Reinforcement Learning from Human F eedback
A short introduction to RLHF and post-training focused on language models.
Nathan Lambert
2 July 2025
Abstract
Reinforcement learning from human feedback (RLHF) has become an important
technical and storytelling tool to deploy the latest machine learning systems. In this
book, we hope to give a gentle introduction to the core methods for people with some
level of quantitative background. The book starts with the origins of RLHF – both
in recent literature and in a convergence of disparate fields of science in economics,
philosophy , and optimal control. W e then set the stage with definitions, problem
formulation, data collection, and other common math used in the literature. The
core of the book details every optimization stage in using RLHF, from starting with
instruction tuning to training a reward model and finally all of rejection sampling,
reinforcement learning, and direct alignment algorithms. The book concludes with


In [6]:
# Step 3: Create embeddings and store in FAISS
embeddings = OllamaEmbeddings(model="mxbai-embed-large")


In [7]:
persissent_directory = "./chroma_db"
vectorstore = Chroma.from_documents(chunks, embeddings, collection_metadata={"hnsw:space": "cosine"}, persist_directory=persissent_directory)

In [8]:
# Step 4: Query and retrieve relevant docs
query = "what is RLHF"
results = vectorstore.similarity_search_with_score(query, k=3)

In [9]:
print("\nTop 3 Relevant Documents:")
for doc, score in results:
    # The score from Chroma is cosine distance (lower is better)
    # We convert it to cosine similarity (higher is better, range 0 to 1)
    similarity_score = 1 - score
    print(f"score: {similarity_score:.4f} content: {doc.page_content}")
    print(f"---------------------------------")


Top 3 Relevant Documents:
score: 0.6309 content: 20.3 Product Cycles, UX, and RLHF
As powerful AI models become closer to products than singular artifacts of an experiment
machine learning process, RLHF has become an interface point for the relationship between
models and product. Much more goes into making a model easy to use than just having
the final model weights be correct – fast inference, suitable tools to use (e.g. search or code
execution), a reliable and easy to understand user interface (UX), and more. RLHF research
has become the interface where a lot of this is tested because of the framing where RLHF
is a way to understand the user’s preferences to products in real time and because it is the
124
---------------------------------
score: 0.6280 content: In many ways, the result is that while RLHF is heavily inspired by RL optimizers and
problem formulations, the action implementation is very distinct from traditional RL.
Figure 3: Standard RLHF loop
4.1.2 Finetuning and Re

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
results = retriever.invoke(query)

print("\n🔍 Top 3 Relevant Documents:")
for doc in results:
    print(doc.page_content)
    print("----------- ----------------------")



🔍 Top 3 Relevant Documents:
20.3 Product Cycles, UX, and RLHF
As powerful AI models become closer to products than singular artifacts of an experiment
machine learning process, RLHF has become an interface point for the relationship between
models and product. Much more goes into making a model easy to use than just having
the final model weights be correct – fast inference, suitable tools to use (e.g. search or code
execution), a reliable and easy to understand user interface (UX), and more. RLHF research
has become the interface where a lot of this is tested because of the framing where RLHF
is a way to understand the user’s preferences to products in real time and because it is the
124
----------- ----------------------
In many ways, the result is that while RLHF is heavily inspired by RL optimizers and
problem formulations, the action implementation is very distinct from traditional RL.
Figure 3: Standard RLHF loop
4.1.2 Finetuning and Regularization
RLHF is implemented from a str

In [11]:
# ---------------------- Query and Retrieve Documents ----------------------
print("\n# ---------------------- Query and Retrieve Documents ----------------------")
query = "what is RLHF?"
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 3, "lambda_mult": 1}
)
results = retriever.invoke(query)

print("\n🔍 Top 3 Relevant Documents:")
for doc in results:
    print(doc.page_content)
    print("----------- ----------------------")

print("\n# ----------------------  lambda = 0.1----------------------")

# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 3, "lambda_mult": 0.1}
)
results = retriever.invoke(query)

print("\n🔍 Top 3 Relevant Documents:")
for doc in results:
    print(doc.page_content)
    print("----------- ----------------------")


# ---------------------- Query and Retrieve Documents ----------------------

🔍 Top 3 Relevant Documents:
20.3 Product Cycles, UX, and RLHF
As powerful AI models become closer to products than singular artifacts of an experiment
machine learning process, RLHF has become an interface point for the relationship between
models and product. Much more goes into making a model easy to use than just having
the final model weights be correct – fast inference, suitable tools to use (e.g. search or code
execution), a reliable and easy to understand user interface (UX), and more. RLHF research
has become the interface where a lot of this is tested because of the framing where RLHF
is a way to understand the user’s preferences to products in real time and because it is the
124
----------- ----------------------
In many ways, the result is that while RLHF is heavily inspired by RL optimizers and
problem formulations, the action implementation is very distinct from traditional RL.
Figure 3: Standar

In [12]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # api_key="...",  # if you prefer to pass api key in directly instaed of using env vars
    # base_url="...",
    # organization="...",
    # other params...
)

In [13]:
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 1}),
    llm=llm
)

multi_results = multi_retriever.get_relevant_documents(query, kwargs={"k": 1})

for i, doc in enumerate(multi_results, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content)
    print()

C:\Users\Prathamesh Bonde\AppData\Local\Temp\ipykernel_30980\3979646991.py:6: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  multi_results = multi_retriever.get_relevant_documents(query, kwargs={"k": 1})


--- Chunk 1 ---
Reinforcement Learning from Human F eedback
A short introduction to RLHF and post-training focused on language models.
Nathan Lambert
2 July 2025
Abstract
Reinforcement learning from human feedback (RLHF) has become an important
technical and storytelling tool to deploy the latest machine learning systems. In this
book, we hope to give a gentle introduction to the core methods for people with some
level of quantitative background. The book starts with the origins of RLHF – both
in recent literature and in a convergence of disparate fields of science in economics,
philosophy , and optimal control. W e then set the stage with definitions, problem
formulation, data collection, and other common math used in the literature. The
core of the book details every optimization stage in using RLHF, from starting with
instruction tuning to training a reward model and finally all of rejection sampling,
reinforcement learning, and direct alignment algorithms. The book concludes with

